In [ ]:
import sys
import matplotlib.pyplot as plt
import csv
import warnings
import numpy as np
import sympy
import matplotlib.cm as cm
import matplotlib as mpl
import os
from pathlib import Path

warnings.filterwarnings("ignore")

# Configuration - modify these as needed
comp_max = 10  # Maximum complexity to consider
downloads_dir = os.path.expanduser("~/Downloads")  # Path to downloads folder
output_dir = os.path.expanduser("~/Downloads/plots")  # Where to save plots
tmax = 5
try_integration = False
xscale = 'linear'
yscale = 'linear'

print('\nMaking plots from all complexities', flush=True)

# Create output directory if it doesn't exist
if not os.path.isdir(output_dir):
    print(f'Creating output directory: {output_dir}')
    os.makedirs(output_dir, exist_ok=True)

# Plot parameters
vmin = 1e-3
vmax = 1

if comp_max >= 8:
    sys.setrecursionlimit(2000 + 500 * (comp_max - 8))

# Collect data from all complexity levels
all_data = []
max_param = 0
complexities_found = []

for c in range(1, comp_max + 1):
    file_path = os.path.join(downloads_dir, f'final_{c}.dat')
    if os.path.exists(file_path):
        print(f"Reading file: {file_path}")
        try:
            with open(file_path, "r") as f:
                reader = csv.reader(f, delimiter=';')
                comp_data = [row for row in reader]
                if len(comp_data) > 0:
                    all_data.extend(comp_data)
                    complexities_found.append(c)
                    # Track maximum number of parameters across all complexities
                    max_param = max(max_param, len(comp_data[0]) - 7)
                    print(f"Found {len(comp_data)} functions at complexity {c}")
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
    else:
        print(f"File {file_path} not found, skipping complexity {c}")

print(f"Mixing results from complexities: {complexities_found}")
print(f"Total functions found: {len(all_data)}")

if len(all_data) == 0:
    print("No functions with finite DL found across all complexities, so will not make figure")
    exit()

fcn_list = [d[1] for d in all_data]
# Pad parameter arrays with zeros for functions with fewer parameters
params = []
for d in all_data:
    param_row = np.array(d[-max_param:], dtype=float) if len(d) >= 7 + max_param else np.zeros(max_param)
    if len(d) < 7 + max_param:
        actual_params = np.array(d[7:], dtype=float) if len(d) > 7 else np.array([])
        param_row[:len(actual_params)] = actual_params
    params.append(param_row)
params = np.array(params)

DL = np.array([d[2] for d in all_data], dtype=float)
if not np.any(np.isfinite(DL)):
    print('All DL are infinite, so skipping plot')
    exit()

DL_min = np.amin(DL[np.isfinite(DL)])
alpha = DL_min - DL
alpha = np.exp(alpha)

# Sort by DL (best functions first) before filtering
finite_mask = np.isfinite(DL)
if np.sum(finite_mask) == 0:
    print('No finite DL values found, so skipping plot')
    exit()

# Sort indices by DL (ascending - best first)
sort_indices = np.argsort(DL[finite_mask])
finite_indices = np.where(finite_mask)[0][sort_indices]

# Apply filtering and sorting
filtered_indices = finite_indices[alpha[finite_indices] > vmin]

fcn_list = [fcn_list[i] for i in filtered_indices]
params = params[filtered_indices,:]
alpha = alpha[filtered_indices]

print(f"After filtering: {len(fcn_list)} functions to plot")

# Create figures
fig = plt.figure(figsize=(7,5))
ax1 = fig.add_axes([0.10,0.10,0.70,0.85])
cmap = cm.hot_r
norm = mpl.colors.LogNorm(vmin=vmin,vmax=vmax)

fig2 = plt.figure(figsize=(7,5))
ax2 = fig2.add_axes([0.10,0.10,0.70,0.85])

# Since we don't have the likelihood object, we'll create some sample data
# You can modify this section to load your actual data
print("Note: Creating sample data since likelihood object is not available")
print("Modify this section to load your actual experimental data")

# Sample x data (you should replace this with your actual data)
x_data = np.linspace(0.1, 10, 50)  # Sample r_proj values
y_data = np.random.exponential(2, 50) + 0.1  # Sample ESD values

# Plot the best functions (limited to 10 for clarity)
num_to_plot = min(len(fcn_list), 10)
for i in range(num_to_plot):
    fcn_i = fcn_list[i].replace('\'', '')
    print(f'{i+1} of {len(fcn_list)}: {fcn_i}')
    
    # For demonstration, create a simple mathematical function
    # In your actual code, you would use your simplifier and likelihood objects
    x_array = np.linspace(0.1, 10, 1000)
    
    # Sample function evaluation (replace with your actual function evaluation)
    # This is just a placeholder - you'll need to adapt this to your actual function processing
    try:
        # Simple example function - replace with your actual function evaluation
        if 'x' in fcn_i:
            # Try to evaluate simple expressions
            safe_fcn = fcn_i.replace('^', '**')  # Convert to Python syntax
            # For safety, only allow basic math operations
            allowed_chars = set('0123456789.+-*/()x abcdefghijklmnopqrstuvwxyz')
            if all(c in allowed_chars for c in safe_fcn.lower()):
                y_pred = eval(safe_fcn.replace('x', 'x_array'))
            else:
                y_pred = np.ones_like(x_array) * (i + 1)  # Fallback
        else:
            y_pred = np.ones_like(x_array) * float(fcn_i) if fcn_i.replace('.','').replace('-','').isdigit() else np.ones_like(x_array) * (i + 1)
    except:
        y_pred = np.ones_like(x_array) * (i + 1)  # Fallback for any errors
    
    # Plot the function
    color = cmap(norm(alpha[i]))
    ax1.plot(x_array, y_pred, color=color, zorder=len(fcn_list)-i, label=f'Func {i+1}')
    ax2.plot(x_array, y_pred, color=color, zorder=len(fcn_list)-i, label=fcn_i[:20])

# Plot sample data points
ax1.plot(x_data, y_data, '.', color='k', ms=5, zorder=len(fcn_list)+1, alpha=1)

# Set labels and formatting
ax1.set_xlabel(r'$r_{proj} (Mpc)$')
ax1.set_ylabel(r'$ESD (10^{12} M_{sun}/Mpc^2)$')
ax1.set_xscale(xscale)
ax1.set_yscale(yscale)
if xscale != 'log':
    ax1.set_xlim(0, None)

ax2.set_xlabel(r'$r_{proj} (Mpc)$')
ax2.set_ylabel(r'Density')
ax2.set_xscale(xscale)
ax2.set_yscale(yscale)

# Add colorbar
ax_cb = fig.add_axes([0.85,0.10,0.05,0.85])
cb1 = mpl.colorbar.ColorbarBase(ax_cb, cmap=cmap, norm=norm, orientation='vertical')
cb1.set_label(r'$\exp \left( MDL - DL \right)$')

# Save figures
fig.tight_layout()
fig.savefig(os.path.join(output_dir, f'plot_all_comps_up_to_{comp_max}.png'), dpi=300, bbox_inches='tight')
print(f"Saved ESD plot to: {os.path.join(output_dir, f'plot_all_comps_up_to_{comp_max}.png')}")

fig2.tight_layout()
fig2.savefig(os.path.join(output_dir, f'density_plot_all_comps_up_to_{comp_max}.png'), dpi=300, bbox_inches='tight')
print(f"Saved density plot to: {os.path.join(output_dir, f'density_plot_all_comps_up_to_{comp_max}.png')}")

plt.show()

print("Plotting completed successfully!")